# 12 · Silver: Central 156 da Emlurb

Transforma os 3 arquivos brutos do 156 (2024, 2025, 2026) numa tabela limpa, uma linha por demanda.

**Papel no produto:** o 156 é uma *alternativa existente* e a fonte do **cold start** do Motor (histórico de onde o problema volta). Não é o MVP. Ver Diretriz-Mestra (pasta 11) e engenharia de dados (pasta 09, seção 2.2).

**Base:** receita de limpeza da análise do 156 (pasta 08), com duas adaptações por causa da versão sanitizada do Bronze:
1. **Sem a coluna NUMERO** (esvaziada pela equipe por ter telefones). A deduplicação usa serviço + logradouro + bairro + data de abertura. Chamados do mesmo serviço, na mesma rua e no mesmo dia, em casas diferentes, viram uma demanda só. Por isso o total fica abaixo das 247.315 demandas de referência (decisão da equipe, 10/11/2026).
2. **Coordenadas estragadas pelo Google Sheets** (ex.: `-80.852.089`). São consertadas por `consertar_coordenada`: tira os pontos e põe o decimal depois do 1º dígito (latitude) e do 2º (longitude).

**Saída:** `data/silver/ocorrencia_156.parquet` (fora do git).

In [1]:
import sys
from pathlib import Path

# Acha a raiz do repositório (a pasta que tem src/) e deixa o pacote wastestack importável
RAIZ = Path.cwd()
while not (RAIZ / "src" / "wastestack").exists() and RAIZ != RAIZ.parent:
    RAIZ = RAIZ.parent
sys.path.insert(0, str(RAIZ / "src"))

In [2]:
import re
import pandas as pd
from wastestack.config import BRONZE, SILVER, LAT_MIN, LAT_MAX, LON_MIN, LON_MAX
from wastestack.limpeza import consertar_coordenada, chave_bairro, normalizar_nome

pd.set_option("display.max_columns", 30)

## 1. Ler os arquivos (tudo como texto) e tirar espaços das pontas

In [3]:
from wastestack.ocorrencias import preparar_156
silver, linhas_brutas = preparar_156()


2024 108082 linhas
2025 86966 linhas
2026 61225 linhas
Total empilhado: 256273
Demandas depois da deduplicação: 230491
Linhas removidas: 25782
Demandas que aparecem em mais de um arquivo: 3530
Bairros distintos: 95
Com coordenada válida: 68640 (29.8%)
Gravado: /sessions/rcw-01r8xt6ghu63xbmmbbhluwpf/mnt/Waste_Stack/data/silver/ocorrencia_156.parquet 230491 linhas


## 2. Deduplicar

Cada arquivo traz as demandas cuja **última movimentação** caiu no ano, então a mesma demanda pode aparecer em mais de um arquivo. Fica a linha do arquivo mais recente e, dentro dele, a de última situação mais recente.

In [4]:
# Etapa executada em wastestack.ocorrencias.preparar_156().


## 3. Padronizar bairro e logradouro

In [5]:
# Etapa executada em wastestack.ocorrencias.preparar_156().


## 4. Consertar e validar as coordenadas

In [6]:
# Etapa executada em wastestack.ocorrencias.preparar_156().


## 5. Recorte WasteStack (lixo, limpeza de vias e drenagem)

Códigos da receita da pasta 08. Ficam de fora: animais mortos, eventos, pintura de meio-fio, conserto de drenagem (obra), pedidos de lixeira ou contêiner e arboviroses. As outras demandas continuam na tabela, com `categoria` vazia.

In [7]:
# Etapa executada em wastestack.ocorrencias.preparar_156().


## 6. Datas, tempo de atendimento e meses completos

In [8]:
# Etapa executada em wastestack.ocorrencias.preparar_156().


## 7. Gravar a Silver

In [9]:
# Etapa executada em wastestack.ocorrencias.preparar_156().


## 8. Conferência com os resultados de referência (pasta 08)

A referência foi calculada **com** a coluna NUMERO na chave de deduplicação. Sem ela, as contagens depois da deduplicação ficam um pouco menores: a diferença é esperada e está registrada abaixo.

In [10]:
recorte = silver[silver["no_recorte"] & silver["periodo_analise"]]
referencia = {
    "linhas empilhadas": (linhas_brutas, 256_273),
    "demandas depois da deduplicação": (len(silver), 247_315),
    "recorte jan/2024–19/06/2026": (len(recorte), 63_449),
    "  lixo e descarte": ((recorte["categoria"] == "Lixo e descarte").sum(), 30_997),
    "  limpeza de vias": ((recorte["categoria"] == "Limpeza de vias").sum(), 20_107),
    "  limpeza de drenagem": ((recorte["categoria"] == "Limpeza de drenagem").sum(), 12_345),
    "recorte nos 25 meses completos": ((recorte["mes_completo"]).sum(), 59_906),
}
tabela = pd.DataFrame(referencia, index=["silver", "referência (com NUMERO)"]).T
tabela["diferença %"] = ((tabela["silver"] / tabela["referência (com NUMERO)"] - 1) * 100).round(1)
tabela

                                 silver  referência (com NUMERO)  diferença %
linhas empilhadas                256273                   256273          0.0
demandas depois da deduplicação  230491                   247315         -6.8
recorte jan/2024–19/06/2026       61790                    63449         -2.6
  lixo e descarte                 29874                    30997         -3.6
  limpeza de vias                 19816                    20107         -1.4
  limpeza de drenagem             12100                    12345         -2.0
recorte nos 25 meses completos    58342                    59906         -2.6


In [11]:
# Mediana de dias até atender, por categoria (referência: 8 / 54 / 91)
recorte.groupby("categoria")["dias_ate_atender"].median()

categoria
Limpeza de drenagem    91.0
Limpeza de vias        54.0
Lixo e descarte         8.0
